# Custom Script to encode datasets and append encoding

In [44]:
import json
import os
from graphMatching.encoders.bf_encoder import BFEncoder
from graphMatching.encoders.tmh_encoder import TMHEncoder
from graphMatching.encoders.tsh_encoder import TSHEncoder
from utils import read_tsv, save_tsv

# Load configuration from dea_config.json
with open('dea_config.json', 'r') as f:
    config = json.load(f)

# Extract the encoding configuration
ENC_CONFIG = config['ENC_CONFIG']
GLOBAL_CONFIG = config['GLOBAL_CONFIG']
GLOBAL_CONFIG["Workers"] = os.cpu_count()

In [29]:
data_dir = "./data/datasets/fakename_1k.tsv"
data, uids, header = read_tsv(data_dir, skip_header=False)

## BF Encoder

In [49]:
# Create BF Encoder using configuration parameters
BFEncoder = BFEncoder(
    secret=ENC_CONFIG["AliceSecret"], 
    filter_size=ENC_CONFIG["AliceBFLength"],
    bits_per_feature=ENC_CONFIG["AliceBits"], 
    ngram_size=ENC_CONFIG["AliceN"], 
    diffusion=ENC_CONFIG["AliceDiffuse"],
    eld_length=ENC_CONFIG["AliceEldLength"], 
    t=ENC_CONFIG["AliceT"]
)

In [50]:
header[-2] = "bloomfilter"
_, alice_data_combined_with_encoding = BFEncoder.encode_and_compare_and_append(data, uids, metric=ENC_CONFIG["AliceMetric"], sim=True,
                                                        store_encs=GLOBAL_CONFIG["SaveAliceEncs"])

In [51]:
save_tsv(alice_data_combined_with_encoding, "./data/datasets/fakename_1k_bf_encoded.tsv", header=header, write_header=True)

## TMH Encoder

In [45]:
TMHEncoder = TMHEncoder(ENC_CONFIG["AliceNHash"], ENC_CONFIG["AliceNHashBits"],
                                    ENC_CONFIG["AliceNSubKeys"], ENC_CONFIG["AliceN"],
                                    ENC_CONFIG["Alice1BitHash"],
                                    random_seed=ENC_CONFIG["AliceSecret"], verbose=GLOBAL_CONFIG["Verbose"],
                                        workers=GLOBAL_CONFIG["Workers"])

In [46]:
header[-2] = "tmh"
_, alice_data_combined_with_encoding = TMHEncoder.encode_and_compare_and_append(data, uids, metric=ENC_CONFIG["AliceMetric"], sim=True, store_encs=GLOBAL_CONFIG["SaveAliceEncs"])

In [47]:
save_tsv(alice_data_combined_with_encoding, "./data/datasets/fakename_1k_tmh_encoded.tsv", header=header, write_header=True)

## TSH Encoder

In [52]:
TSHEncoder = TSHEncoder(ENC_CONFIG["AliceNHashFunc"], ENC_CONFIG["AliceNHashCol"], ENC_CONFIG["AliceN"],
                                    ENC_CONFIG["AliceRandMode"], secret=ENC_CONFIG["AliceSecret"],
                                    verbose=GLOBAL_CONFIG["Verbose"], workers=GLOBAL_CONFIG["Workers"])

In [53]:
header[-2] = "tsh"
_, alice_data_combined_with_encoding = TSHEncoder.encode_and_compare_and_append(data, uids, metric=ENC_CONFIG["AliceMetric"], sim=True, store_encs=GLOBAL_CONFIG["SaveAliceEncs"])

In [54]:
save_tsv(alice_data_combined_with_encoding, "./data/datasets/fakename_1k_tsh_encoded.tsv", header=header, write_header=True)